In [57]:
#####################################################################################################################
#####################################################################################################################

import random


def generate_r():
    r = random.random()

    if r < 0.25:
        return [round(random.uniform(1.0, 1.5),2), round(random.uniform(0.6, 0.8), 2)]
    elif r < 0.5:
        return [round(random.uniform(1.5, 2.0), 2), round(random.uniform(0.5, 0.7), 2)]
    elif r < 0.75:
        return [round(random.uniform(2.0, 2.5), 2), round(random.uniform(0.4, 0.6), 2)]
    else:
        return [round(random.uniform(2.5, 3.0), 2), round(random.uniform(0.3, 0.5), 2)]


###盈亏比R与P按照预定值下的分段式随机生成器

lst = []
n = 10

for i in range(n):
    R = generate_r()
    lst.append(R)
###建立（r,p）值list


profit_list = [row[0] for row in lst]
R_list = [row[1] for row in lst]
list_value = list(map(lambda rp: round(rp[1]*rp[0] - (1 - rp[1]),2), lst))

list_items=list(range(1,len(list_value)+1))

list_weight=[]
for i in range(1,len(list_value)+1):
    list_weight.append(round(random.uniform(5, 20), 2))

#####################################################################################################################

import pandas as pd
#from BB import list_weight,list_items,list_value,R_list
#if not (len(list_items) == len(list_weight) == len(list_value)):
    #raise ValueError("list_items, list_weight, list_value 长度必须一致")
W = 100
#####################################################################################################################
#####################################################################################################################



class Item:
    def __init__(self,id,w,v):
        self.id = id
        self.w = w
        self.v = v
        self.ratio = v / w
#list_value_weight=list(map(lambda x,y:x/y,list_value,list_weight))

list_items_final=[]
for id,w,v in zip(list_items,list_weight,list_value):
    list_items_final.append(Item(id,w,v))

list_items_final.sort(key=lambda x:x.ratio,reverse=True)
##对列表对象按照价值/重量比由大到小排列
#print("---------序列物：（id,w,v）---------")
#for Item in list_items_final:
    #print(Item.id,",",Item.w,",",Item.v)







class Node:
    def __init__(self, level, cw, cv,ub=0,choose=None,parent=None):
        #设定了Node节点类型，它的对象是node，初始节点使用root = Node(-1, 0, 0)的方式表现。由于大量使用三个参数的node
        self.level = level
        self.cw = cw
        self.cv = cv
        self.ub = ub
        self.choose = choose
        self.parent = parent





##########生成器函数，用于yield两个child，对应放入背包或不放入背包的情况
def expand(node):
    next_level = node.level + 1
    if next_level >= len(list_items_final):
        return

    item = list_items_final[next_level]


    put_in_bag = Node(next_level, node.cw + item.w, node.cv + item.v,choose=item,
        parent=node)
    yield put_in_bag


    not_put_in_bag = Node(next_level, node.cw, node.cv,choose=None,
        parent=node)
    yield not_put_in_bag


##对节点ub值进行计算的函数
def bound(node):
    if node.cw >= W:
        return 0
    if node.cw == W:
        return node.cv

    profit = node.cv
    weight = node.cw
    #设置两个参数表示本节点已累积的价值和重量

    i = node.level + 1
    #进入本节点的下一层

    while i < len(list_items_final) and weight + list_items_final[i].w <= W:
        #while循环中，从下一层的物件开始放入背包直到放完物件或背包到达重量上限
        weight += list_items_final[i].w
        profit += list_items_final[i].v
        #在节点累积的基础上不断累积每次新增的重量和价值
        i += 1

    if i < len(list_items_final):
        #while循环结束后，如果仍然有物件没有放入背包
        profit += (W - weight) * list_items_final[i].v / list_items_final[i].w
        #获得了剩余背包容量（若有）和下一个未能放入的物件的V/M比率。计算了本节点的ub值。
    return profit
#使用profit作为ub计算的变量名，等待绑定




def branch_and_bound():
    root = Node(-1, 0, 0)
    root.ub = bound(root)
    #绑定了初始节点的ub

    best = 0
    best_node = root
    waiting_nodes = [root]
    #用于存储有效节点的列表，目前已存入初始节点了

    while waiting_nodes:
        node = waiting_nodes.pop()
        #取出列表最新的有效节点

        # 如果这个节点ub都小于等于 best，剪枝
        if node.ub <= best:
            continue

        for child in expand(node):
        #调用生成器函数expand，投入目前节点。目前节点被生成器函数生成了两个预选的潜在节点（意味着物件放入背包或不放入背包）

            if child.cw > W:
                continue

            # 更新当前最优值
            if child.cv > best:
                best = child.cv
                best_node = child


            # 估计上界ub值，尝试下一次选择
            child.ub = bound(child)

            # 如果结果超过 best，加入有效节点列表成为一个有效节点
        ##如果两个child判定都通过，那么会出现两个node放入有效节点列表
            if child.ub > best:
                waiting_nodes.append(child)


    return best, best_node
#返回本次更新的best值，这是对child检定后更新的best值

answer, best_node = branch_and_bound()
#print("最优值 =", round(answer,3))

path = []
current = best_node

while current is not None:
    if current.choose is not None:
        path.append(current.choose)
    current = current.parent

path.reverse()

#print("---------最优路径:（id,w,v）---------")
#for item in path:
    #print(item.id, ",", item.w, ",", item.v)


#####################################################################################################################
#####################################################################################################################

def greedy_knapsack(items, W):
    selected_items = []
    total_weight = 0
    total_value = 0

    for item in items:
        if total_weight + item.w <= W:
            selected_items.append(item)
            total_weight += item.w
            total_value += item.v

    return selected_items, total_weight, total_value
selected_items, total_weight, total_value = greedy_knapsack(list_items_final, W)

#####################################################################################################################


#print("---------贪心路径:（id,w,v）---------")
#for item in selected_items:
    #print(item.id, ",", item.w, ",", item.v)

#print("贪心总重量 =", round(total_weight, 2))
#print("贪心总价值 =", round(total_value, 2))
#print("最优值 =", round(answer,3))
#print(abs(answer - total_value) < 1e-6)

#####################################################################################################################
#####################################################################################################################
#####################################################################################################################
import pandas as pd
from IPython.display import display, Markdown

items_df = pd.DataFrame({
    "物件": list_items,
    "w": list_weight,
    "v": list_value
})

display(Markdown("## 初始物件表"))
display(
    items_df.style
    .format({
        "w": "{:.2f}",
        "v": "{:.2f}"
    })
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "center")]},
    ])
)

#####################################################################################################################
import pandas as pd
from IPython.display import display, Markdown

summary_df = pd.DataFrame({
    "指标": [
        "贪心法总重",
        "贪心法总价值",
        "分支法总重",
        "分支法总价值",
        "是否一致"
    ],
    "结果": [
        round(total_weight,2),
        total_value,
        sum(item.w for item in path),
        round(answer,2),
        "T" if abs(answer - total_value) < 1e-6 else "False"
    ]
})

display(Markdown("## 关键数据"))
display(
    summary_df.style
    .format({
        "结果": lambda x: f"{x:.3f}" if isinstance(x, (int, float)) else str(x)
    })
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "center")]},
    ])
)

#####################################################################################################################

import pandas as pd
from IPython.display import display, Markdown

def build_path_table(path):
    rows = []
    cum_w = 0
    cum_v = 0

    for item in path:
        cum_w += item.w
        cum_v += item.v
        rows.append({
            "path_node": item.id,
            "w": item.w,
            "v": item.v,
            "cum_w": cum_w,
            "cum_v": cum_v
        })

    return pd.DataFrame(rows)

def show_table(title, path):
    df = build_path_table(path)

    styled = (
        df.style
        .format({
            "w": "{:.2f}",
            "v": "{:.2f}",
            "cum_w": "{:.2f}",
            "cum_v": "{:.2f}",
        })
        .hide(axis="index")
        .set_properties(**{
            "text-align": "center"
        })
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "center")]},
            {"selector": "td", "props": [("text-align", "center")]},
        ])
    )

    display(Markdown(f"## {title}"))
    display(styled)

show_table("Branch and Bound Algorithm分支界定法", path)g
show_table("Greedy Algorithm贪心算法", selected_items)

#####################################################################################################################
#####################################################################################################################
#####################################################################################################################





## 初始物件表

物件,w,v
1,13.63,0.63
2,12.83,0.63
3,14.94,0.58
4,17.21,0.73
5,8.76,0.51
6,19.65,0.86
7,12.83,0.40
8,11.73,0.71
9,5.13,0.58
10,8.64,0.32


## 关键数据

指标,结果
贪心法总重,97.580
贪心法总价值,4.970
分支法总重,97.580
分支法总价值,4.970
是否一致,T


## Branch and Bound Algorithm分支界定法

path_node,w,v,cum_w,cum_v
9,5.13,0.58,5.13,0.58
8,11.73,0.71,16.86,1.29
5,8.76,0.51,25.62,1.80
2,12.83,0.63,38.45,2.43
1,13.63,0.63,52.08,3.06
6,19.65,0.86,71.73,3.92
4,17.21,0.73,88.94,4.65
10,8.64,0.32,97.58,4.97


## Greedy Algorithm贪心算法

path_node,w,v,cum_w,cum_v
9,5.13,0.58,5.13,0.58
8,11.73,0.71,16.86,1.29
5,8.76,0.51,25.62,1.80
2,12.83,0.63,38.45,2.43
1,13.63,0.63,52.08,3.06
6,19.65,0.86,71.73,3.92
4,17.21,0.73,88.94,4.65
10,8.64,0.32,97.58,4.97
